In [ ]:
# Import necessary packages
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from mpl_toolkits.basemap import Basemap
import osmnx as ox

from statsmodels.formula.api import ols
from statsmodels.multivariate.manova import MANOVA
import statsmodels.api as sm
from scipy.optimize import curve_fit
from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.feature_selection import f_regression

from sampling_rates import PFAS_DATA

In [ ]:
### initialize data and colors for plotting

# read in site information
sites_df = pd.read_csv('sites.csv', index_col=0)

# reference data from papers
# ground water in Australia
sampling_rates_australia_ground = {'Mean Rs [mL/day]': {
    'PFBA': 1.95, 'PFPeA': 2.66, 'PFHxA': 2.95, 'PFHpA': 2.88, 'PFOA': 2.87,
    'PFBS': 3.12, 'PFHxS': 3.29, 'PFOS': 4.7, '6:2 FTS': 2.78,
    }, 'Std Rs [mL/day]': {
    'PFBA': 0.26, 'PFPeA': 0.53, 'PFHxA': 0.77, 'PFHpA': 0.49, 'PFOA': 0.38,
    'PFBS': 0.64, 'PFHxS': 0.64, 'PFOS': 0.31, '6:2 FTS': 0.26,
    },
    }

# fresh water in spain and australia
sampling_rates_ref_fresh = {'spain': {'reference Mean Rs [mL/(day cm)]': {
    'PFBA': 3.8, 'PFPeA': 5.2, 'PFHxA': 4.825, 'PFHpA': 5.95, 'PFOA': 4.7, 'PFNA':6, 'PFDA': 7.325,
    'PFBS': 7.1, 'PFHxS': 5.5, 'PFOS': 4.65, '6:2 FTS': 5.425,
    }}, 'australia': {'reference Mean Rs [mL/(day cm)]': {
    'PFHxA': 5.5, 'PFHpA': 8.0, 'PFOA': 6.0, 'PFNA': 3.25, 'PFDA': 1.875,
    'PFHxS': 6.0, 'PFOS': 2.375, 'FBSA': 6.5,
    }}}

# define colors
water_type_colors = {
    'fresh': "#0d8631",
    'salt':"#00dbe3",
    'ground':"#B70BF5",
}

temperature_level_colors = {
    'warm': "#A90D15",
    'moderate':"#62128a",
    'cold':"#1A0BF5",
}

plot_colors = [water_type_colors[water_type] for water_type in sites_df['Water Type']]

# Boxplot styling: open markers, different outlier style, different color
boxplot_props = dict(
    patch_artist=False,
    boxprops=dict(color='darkorange'),
    medianprops=dict(color='darkorange'),
    whiskerprops=dict(color='darkorange'),
    capprops=dict(color='darkorange'),
    flierprops=dict(marker='.', markerfacecolor='none',
                    markeredgecolor='gray', markersize=6, linestyle='none'),
    grid=False,
)

# define function for legend markers
wtlp = lambda i: plt.plot(
    [], color=water_type_colors[i], mec="none", label=i, ls="", marker='*'
    )[0]

wtlpax = lambda i, ax: ax.plot(
    [], color=water_type_colors[i], mec="none", label=i, ls="", marker='*'
    )[0]

# initialize pfas data class
pfas_data_class = PFAS_DATA()

In [ ]:
### plot sampling locations

# define bounding box coordinates: south, north, west, east
north, south, east, west = 41.3, 41.9, -71.9, -70.4

# make map
fig = plt.figure()
fig.set_size_inches([17.05,10.15])
ax = fig.add_subplot(111)

# plot basemap, rivers and countries
m = Basemap(llcrnrlat=north, urcrnrlat=south, llcrnrlon=east, urcrnrlon=west, resolution='f')
m.arcgisimage(service='World_Shaded_Relief')

# query for waterways
bounding = tuple([west, south, east, north])
rivers = ox.features_from_bbox(bbox=bounding, tags={'waterway': True,})
lakes = ox.features_from_bbox(bbox=bounding, tags={'water': True})

# plot
rivers.plot(color='dodgerblue', ax=ax)
lakes.plot(color='dodgerblue', ax=ax)
m.scatter(sites_df['Longitude, in decimal degree'].to_list(), sites_df['Latitude, in decimal degree'].to_list(), marker = 'o', color=plot_colors)
for (site_code, sample_loc) in sites_df.iterrows():
    x = sample_loc['Longitude, in decimal degree']
    y = sample_loc['Latitude, in decimal degree']
    xx, yy = m(x, y) #m is basemap object
    if site_code =='WL1':
        plt.text(xx, yy, site_code, fontsize=10, va='bottom', ha='center',)
    elif site_code == '3' or site_code == 'FPB' or site_code == 'BP':
        plt.text(xx, yy, site_code, fontsize=10, va='bottom', ha='left',)
    elif 'WL2' in site_code or 'G1' in site_code or 'G3' in site_code:
        plt.text(xx, yy, site_code, fontsize=10, va='bottom', ha='left',)
    else:
        plt.text(xx,yy, site_code, fontsize=10, va='top', ha='right',)
plt.show()
fig.savefig('plots/sampling_locations.png')

In [ ]:
### read in combined data and filter for further analysis
# write data from grab samples to csv
pe_tube = pd.read_csv('pe_tube.csv', parse_dates=['Deployment Date', 'Recovery Date'])
pe_tube['Deployment Time'] = pd.to_timedelta(pe_tube['Deployment Time'])

pe_tube['Concentration Time'] = pe_tube['Deployment Time'].dt.days * pe_tube['Grab Sample Average']
pe_tube = pe_tube.loc[pe_tube['Concentration Time'] > 5, :]

# more background info to 70 mL/day filter if needed
print(np.percentile(pe_tube['Sampling Rate Measured [mL/day]'], 97.5))
plt.hist(pe_tube['Sampling Rate Measured [mL/day]'], bins=300, label="Histogram of Sampling Rate Measured [mL/day]")
plt.plot([70,70], [0, 80], color='red', linestyle='--', linewidth=2, label='70 mL/day threshold')
plt.legend()
plt.xlabel('Sampling Rate Measured [mL/day]')
plt.ylabel('Detection Frequency')
plt.show()
pe_tube.loc[pe_tube['Sampling Rate Measured [mL/day]'] >= 70, :].to_csv('test.csv')

# filter 'dangerous' data points
pe_tube = pe_tube.loc[pe_tube['Sampling Rate Measured [mL/day]'] < 70, :]
pe_tube['Sampling Rate Measured [mL/(day cm)]'] = pe_tube['Sampling Rate Measured [mL/day]'] / pfas_data_class.effective_length

# sort pe tube data according to compound order
custom_order = pfas_data_class.pfas_data.index.to_list()
pe_tube['Compound'] = pd.Categorical(
    pe_tube['Compound'], categories=custom_order, ordered=True
    )
pe_tube = pe_tube.sort_values('Compound')

In [ ]:
### initialize dataframe to store fitting parameters
indices = []
for pfas_compound in pe_tube['Compound'].unique():
    for water_type in ['fresh', 'salt']:
        indices.append((pfas_compound, water_type))

parameters_fresh_salt = pd.DataFrame(
    columns = [
        'Mean Rs [mL/(day cm)]', 'Std Rs [mL/(day cm)]', 'Number of points',
        'inverse linear slope [mL/(ng day cm)]', 'inverse Std slope [mL/(ng day cm)]', 'inverse linear intercept [mL/(day cm)]',
        'inverse Std intercept [mL/(day cm)]', 'Delta inverse Rs [mL/(day cm)]', 'inverse linear p-value',
        'Rs exponential coefficient [mL/(day cm)]', 'Std exponential coefficient [mL/(day cm)]',
        'log(KD) pre-exponential factor [mL/(g cm)]', 'log(Std exponential coefficient) [mL/(g cm)]',
        'Rs15 [mL/(day cm)]', 'Delta Rs15 [mL/(day cm)]',
        'average R2score', 'average MAE [mL/(day cm)]', 'average RMSE [mL/(day cm)]',
        'inverse linear R2score', 'inverse linear MAE [mL/(day cm)]', 'inverse linear RMSE [mL/(day cm)]',
        'exponential R2score', 'exponential MAE [mL/(day cm)]', 'exponential RMSE [mL/(day cm)]',
        'model R2score', 'model MAE [mL/(day cm)]', 'model RMSE [mL/(day cm)]',
        'ref Spain Mean Rs [mL/(day cm)]', 'ref Australia Mean Rs [mL/(day cm)]',
        'coefficient of variation [%]',
        ],
    index = pd.MultiIndex.from_tuples(indices)
    )

In [ ]:
### function to calculate confidence intervals for sklearn linear regression coefficients

def get_conf_int(alpha, lr, X, y):
    
    """
    Returns (1-alpha) 2-sided confidence intervals
    for sklearn.LinearRegression coefficients
    as a pandas DataFrame
    """
    X_aux = X.copy()
    X_aux.insert(0, 'const', 1)
    dof = -np.diff(X_aux.shape)[0]
    mse = (np.sum((y - pd.DataFrame(lr.predict(X))) ** 2) / dof).to_list()[0]
    var_params = np.diag(np.linalg.inv(X_aux.T.dot(X_aux)))
    t_val = stats.t.isf(alpha/2, dof)
    gap = t_val * np.sqrt(mse * var_params)

    return gap

In [ ]:
# function to assign number of points to xlabel in brackets
def add_counts_to_xticklabels(ax, counts, rotation=90, strip_chars="(),"):
    """
    Append data point counts in brackets to existing x-tick labels.
    
    Parameters
    ----------
    ax : matplotlib.axes.Axes
        The axis whose x-tick labels should be modified.
    counts : dict or list
        If dict: maps compound name (str) -> count (int).
        If list: counts in the same order as the x-ticks.
    rotation : float, optional
        Rotation angle for the new tick labels (default 90).
    strip_chars : str, optional
        Characters to strip from the original tick label text before lookup
        (useful for pandas boxplots which wrap labels like "(FOSA,)").
    
    Returns
    -------
    list of str
        The new tick labels that were applied.
    """
    new_labels = []
    for i, (x_tick, label) in enumerate(zip(ax.get_xticks(), ax.get_xticklabels())):
        key = label.get_text().strip(strip_chars)
        
        if isinstance(counts, dict):
            n = counts.get(key, '')
        else:  # list / array / Series
            try:
                n = counts[i]
            except (IndexError, KeyError):
                n = ''
        
        new_labels.append(f'{key} [n={n}]' if n != '' else key)
    
    ax.set_xticks(ax.get_xticks())  # avoids FixedLocator warning
    ax.set_xticklabels(new_labels, rotation=rotation)
    return new_labels

In [ ]:
### analysis and plots for ground water data
def estimate_kWBL(compound: str) -> list[float, float]:
    if compound in [
        'PFPrA', 'PFBA', 'PFPeA', 'PFHxA','PFHpA', 'PFOA',
        'PFBS', 'PFPeS','PFHxS', 'PFHpS', '6:2 FTS','PFECHS'
        ]:
        return([2e-6, 2e-6])
    if compound in ['PFNA','PFDA', 'PFOS']:
        return([5e-6, 2.5e-6])
    if compound in ['FHpSA', ]:
        return([1e-5, 5e-6])
    if compound in ['FOSA']:
        return([np.nan, np.nan])
    print(f"Compound {compound} not defined")
    return[np.nan, np.nan]

# define compounds relevant for further analysis of ground water
# predefined, because also data from Australia is relevant
keep_categories = [
      'PFPrA', 'PFBA', 'PFPeA', 'PFHxA', 'PFHpA', 'PFOA', 'PFNA', 'PFDA',
      'PFBS', 'PFPeS', 'PFHxS', 'PFHpS', 'PFOS', '6:2 FTS', 'FHpSA', 'FOSA', 'PFECHS'
      ]

parameters_ground = pd.DataFrame(
    columns = [
          'Mean Rs [mL/(day cm)]', 'Std Rs [mL/(day cm)]', 'Mean k_WBL [cm/s]', 
          'reference Mean Rs [mL/(day cm)]', 'reference Std Rs [mL/(day cm)]', 'reference k_WBL [cm/s]',
          'cluster k_WBL [cm/s]'
          ], index = pd.Index(keep_categories, name='Compound')
)

for compound, _ in parameters_ground.iterrows():
    if compound in sampling_rates_australia_ground['Mean Rs [mL/day]'].keys():
        parameters_ground.at[compound, 'reference Mean Rs [mL/(day cm)]'] = \
            round(sampling_rates_australia_ground['Mean Rs [mL/day]'][compound] / 4, 1)
        parameters_ground.at[compound, 'reference Std Rs [mL/(day cm)]'] = \
            round(sampling_rates_australia_ground['Std Rs [mL/day]'][compound] / 4, 1)
        parameters_ground.at[compound, 'reference k_WBL [cm/s]'] = round(
            pfas_data_class.determine_kWBL_from_sampling_rate(
            compound=compound, temperature_in_C=22, Rs=sampling_rates_australia_ground['Mean Rs [mL/day]'][compound],
            area_correction=4/5
            ), 7)
                
# extract and prepare ground water data
pe_tube_ground = pe_tube.loc[pe_tube['Water Type'] == 'ground', :]
pe_tube_ground['Compound'] = pe_tube_ground['Compound'].cat.set_categories(keep_categories, )
pe_tube_ground['Mean k_WBL [cm/s]'] = np.nan

# set boxplot color to groundwater color
boxplot_props['boxprops']['color'] = water_type_colors['ground']
boxplot_props['medianprops']['color'] = water_type_colors['ground']
boxplot_props['whiskerprops']['color'] = water_type_colors['ground']
boxplot_props['capprops']['color'] = water_type_colors['ground']

# evaluate water boundary layer
for index, row in pe_tube_ground.iterrows():
    if not pd.isna(row['Compound']):
        pe_tube_ground.at[index, 'k_WBL [cm/s]'] = pfas_data_class.determine_kWBL_from_sampling_rate(
            compound=row['Compound'], temperature_in_C=row['Average Temperature in C'],
                Rs=row['Sampling Rate Measured [mL/day]']
                )

# prepare information for plots
number_of_points = pe_tube_ground.groupby(['Compound'])['Sampling Rate Measured [mL/(day cm)]'].size()
number_of_points_kWBL = pe_tube_ground.groupby(['Compound'])['k_WBL [cm/s]'].count()
compound_indices = number_of_points.index.tolist()
mean_kWBL = pe_tube_ground.groupby(['Compound'])['k_WBL [cm/s]'].mean()

for compound in pe_tube_ground['Compound'].unique():
    pe_tube_compound_component = pe_tube_ground.loc[pe_tube_ground['Compound'] == compound, :]
    parameters_ground.at[compound, 'Mean Rs [mL/(day cm)]'] = round(pe_tube_compound_component['Sampling Rate Measured [mL/(day cm)]'].mean(), ndigits=1)
    parameters_ground.at[compound, 'Std Rs [mL/(day cm)]'] = round(pe_tube_compound_component['Sampling Rate Measured [mL/(day cm)]'].std(), ndigits=1)

parameters_ground['Mean k_WBL [cm/s]'] = round(mean_kWBL, ndigits=7)

fig_Rs, ax_Rs = plt.subplots()
fig_WBL, ax_WBL = plt.subplots()

pe_tube_ground.boxplot(column='Sampling Rate Measured [mL/(day cm)]', by='Compound', ax=ax_Rs, **boxplot_props)
pe_tube_ground.boxplot(column='k_WBL [cm/s]', by='Compound', ax=ax_WBL, **boxplot_props)

# plot model
compound_indices = [elem for (elem, flag) in zip(compound_indices, number_of_points) if flag !=0]
sampling_rate_model_15, delta_sampling_rate_model_15 = pfas_data_class.evaluate_sampling_rates_compound_list(
    compound_list=compound_indices, temperature_in_C=15, deltaT=10, area_correction=1/5
)
ax_Rs.plot(
    ax_Rs.get_xticks(), np.array(sampling_rate_model_15) + np.array(delta_sampling_rate_model_15),
    color='k', linestyle='dotted', label='R_S + $\\Delta$ R_S'
    )
ax_Rs.plot(
    ax_Rs.get_xticks(), np.array(sampling_rate_model_15) - np.array(delta_sampling_rate_model_15),
    color='k', linestyle='dotted', label='R_S - $\\Delta$ R_S'
    )

for ax in [ax_Rs, ax_WBL]:
    for (x_tick, x_tick_label) in zip(ax.get_xticks(), ax.get_xticklabels()):
        pfas_key = x_tick_label.get_text().strip("(),")
        if ax == ax_Rs:
            if mean_kWBL[x_tick - 1] > 0:
                sampling_rate_model, delta_sampling_rate_model = pfas_data_class.evaluate_sampling_rate(
                    compound=pfas_key, temperature_in_C=17, k_WBL=mean_kWBL[x_tick - 1],
                    delta_kWBL=mean_kWBL[x_tick - 1] * 0.1, deltaT=5, area_correction=1/5,
                    )
                # ax.errorbar(x_tick, y=sampling_rate_model, yerr=delta_sampling_rate_model, color='green', marker='x')
            if pfas_key in sampling_rates_australia_ground['Mean Rs [mL/day]'].keys():
                ax.scatter(
                    x=x_tick, y=parameters_ground.at[pfas_key, 'reference Mean Rs [mL/(day cm)]'],
                    color='blue', marker='x', label='Australia'
                    )
                sampling_rate_model, delta_sampling_rate_model = pfas_data_class.evaluate_sampling_rate(
                    compound=pfas_key, temperature_in_C=22, k_WBL=parameters_ground.at[pfas_key, 'reference k_WBL [cm/s]'],
                    delta_kWBL=parameters_ground.at[pfas_key, 'reference k_WBL [cm/s]'] * 0.1, deltaT=5, area_correction=1/5
                    )
                # ax.errorbar(x=x_tick, y=sampling_rate_model, yerr=delta_sampling_rate_model, color='red', marker='x')
            ax.scatter(
                x=x_tick, y=parameters_ground.at[pfas_key, 'Mean Rs [mL/(day cm)]'], facecolors='none',
                edgecolors=water_type_colors['ground'], s=40, linewidth=1.5,
                )
        else:
            ax.scatter(
                x=x_tick, y=parameters_ground.at[pfas_key, 'reference k_WBL [cm/s]'],
                color='blue', label='Australia', marker='x'
                )
            kWBL, delta_kWBL = estimate_kWBL(compound=pfas_key)
            ax.errorbar(x=x_tick, xerr=0.4, y=kWBL, yerr=delta_kWBL, color='black')
            parameters_ground.at[pfas_key, 'cluster k_WBL [cm/s]'] = round(kWBL, 7)
    box = ax.get_position()
    ax.set_xlabel('')
    ax.set_title('') 
    ax.yaxis.grid(True)

#counts_dict_Rs = dict(zip(pe_tube_ground['Compound'].unique(), number_of_points))
add_counts_to_xticklabels(ax_Rs, number_of_points, rotation=90)
add_counts_to_xticklabels(ax_WBL, number_of_points_kWBL, rotation=90)

legend_elements = [
    Line2D([0], [0], marker='o', markersize=6, markerfacecolor='none',
            markeredgecolor=water_type_colors['ground'], markeredgewidth=1.5,
            linestyle='none', label='Mean'),
    Line2D([0], [0], marker='x', color='w', 
        markeredgecolor='blue', label='Reference Australia'),
    Patch(edgecolor=water_type_colors['ground'], fill=False, label='Boxplot'),
    Line2D([0], [0], marker='.', color='w', markeredgecolor='gray', markersize=6, linestyle='none',
            label='Outlier',),
    Line2D([0], [0], color='k', linestyle='dotted', label='Model bounds' + '\n' + 'neglecting WBL',),
            ]
ax_Rs.legend(
    handles=legend_elements, loc='center', fancybox=True, ncol=2
    )
ax_Rs.set_ylabel('Sampling Rate [mL/(day cm)]')
ax_WBL.set_ylabel('Transport Coefficient WBL [cm/s]')
ax_Rs.set_ylim([0, 6])
ax_WBL.set_ylim([0, 2.2e-5])
fig_Rs.suptitle('')
fig_WBL.suptitle('')
plt.tight_layout()
fig_Rs.savefig('plots/ground_sampling_rate.png', bbox_inches='tight', pad_inches=0)
fig_WBL.savefig('plots/ground_water_boundary.png', bbox_inches='tight', pad_inches=0)

parameters_ground.to_csv('parameters_ground.csv')

In [ ]:
# challenge for ground water deployments:

fig, ax = plt.subplots()

colors = plt.cm.tab20(np.linspace(0, 1, len(keep_categories)))  # 20 distinct colors

compound_legend = lambda i: plt.plot(
    [], color=colors[i], mec="none", label=keep_categories[i], ls="-",
    )[0]

for color, compound in zip(colors, keep_categories):
    kWBL, delta_kWBL = estimate_kWBL(compound=compound)
    if compound in sampling_rates_australia_ground['Mean Rs [mL/day]'].keys():
        sampling_rate_model, delta_sampling_rate_model = pfas_data_class.evaluate_sampling_rate(
            compound=compound, temperature_in_C=22, k_WBL=kWBL, delta_kWBL=delta_kWBL, deltaT=5, area_correction=1/5
            )
        ax.errorbar(
            x=sampling_rates_australia_ground['Mean Rs [mL/day]'][compound] / 4, xerr=sampling_rates_australia_ground['Std Rs [mL/day]'][compound] / 4,
            y=sampling_rate_model, yerr=delta_sampling_rate_model,
            color=color, label=compound, marker='x'
        )
    
    mean_Rs = parameters_ground.loc[compound, 'Mean Rs [mL/(day cm)]']
    std_Rs = parameters_ground.loc[compound, 'Std Rs [mL/(day cm)]']
    sampling_rate_model, delta_sampling_rate_model = pfas_data_class.evaluate_sampling_rate(
        compound=compound, temperature_in_C=17, k_WBL=kWBL, delta_kWBL=delta_kWBL, deltaT=5, area_correction=1/5
        )
    eb = ax.errorbar(
        x=mean_Rs, xerr=std_Rs, y=sampling_rate_model, yerr=delta_sampling_rate_model,
        color=color, label=compound,
    )
    eb[-1][0].set_linestyle('--')
    eb[-1][1].set_linestyle('--')
    if compound == 'FOSA':
        print(mean_Rs, std_Rs, sampling_rate_model, delta_sampling_rate_model)
ax.plot([0, 3], [0, 3], color='black', ls='--')
ax.legend(handles=[compound_legend(index) for index in range(len(keep_categories))], loc='upper center', ncol=4)
ax.set_xlabel('Measured Rs [mL/(day cm)]')
ax.set_ylabel('Modeled Rs [mL/(day cm)]')
ax.set_xlim([0,3])
ax.set_ylim([-0.5,5])

plt.tight_layout()
fig.savefig('plots/ground_challenge.png')

# Fit cumulative sampling volume over deployment time with exponential function

$$
c_s = K_{sw} \cdot c_w (1-\exp(-\frac{R_s \cdot t}{K_{sw}})) \\

\frac{c_s}{c_w} = a \cdot (1-\exp(\frac{-b \cdot t}{a}))
$$

# Inverse linear fit for sampling rate over water concentration times deployment time
$$
R_S = \frac{N_S' + offset}{c_W \cdot \tau} = R_S' + \frac{offset}{c_W \cdot \tau}
$$

In [ ]:
# function for exponential model
concentration_exponential = lambda time, a, b: b * (pfas_data_class.sorbent_weight /pfas_data_class.effective_length) * \
      (1 - np.exp((-1) * a * time / (pfas_data_class.sorbent_weight * b / pfas_data_class.effective_length)))

# initialize figure and axis for plotting
fig, ax = plt.subplots()

# loop over compounds and water types
for compound in pe_tube['Compound'].unique():
    # extract data for selected compound
    compound_data = pe_tube.loc[pe_tube['Compound'] == compound, :]
    # convert deployment time to integer days
    compound_data.loc[:,'Deployment Time [Days]'] = compound_data.loc[:,'Deployment Time'].dt.days
    # make sure compound name does not include special characters
    compound_name_plot = compound.replace(':', '_')
    compound_name_plot = compound_name_plot.replace(' ', '')

    # calculate modelled sampling rates for T=15 p/m 10 degrees
    sampling_rate, delta_sampling_rate = pfas_data_class.evaluate_sampling_rate(
        compound=compound, temperature_in_C=15, deltaT=10, area_correction=1/5
        )

    # loop over water types
    for water_type in ['fresh', 'salt']:
        compound_water_type_data = compound_data.loc[compound_data['Water Type'] == water_type, :]
        # extract deployment time (x) and cs_cw_ratio (cumulative sampling volume)
        time = (compound_water_type_data['Deployment Time'].dt.days).to_numpy()
        cs_cw_ratio = (
            compound_water_type_data['Sampling Rate Measured [mL/(day cm)]'] * compound_water_type_data['Deployment Time'].dt.days
            ).to_numpy()
        if len(time) >= 1:
            # mean and std of sampling rate determined from field deployments
            sampling_rate_data_mean = compound_water_type_data['Sampling Rate Measured [mL/(day cm)]'].mean()
            sampling_rate_data_std = compound_water_type_data['Sampling Rate Measured [mL/(day cm)]'].std()
            mean_sampling_rate_r2 = r2_score(
                compound_water_type_data['Sampling Rate Measured [mL/(day cm)]'],
                sampling_rate_data_mean * np.ones(len(compound_water_type_data))
            )
            mean_sampling_rate_mae = mean_absolute_error(
                compound_water_type_data['Sampling Rate Measured [mL/(day cm)]'],
                sampling_rate_data_mean * np.ones(len(compound_water_type_data))
            )
            mean_sampling_rate_rmse = root_mean_squared_error(
                compound_water_type_data['Sampling Rate Measured [mL/(day cm)]'],
                sampling_rate_data_mean * np.ones(len(compound_water_type_data))
            )
            parameters_fresh_salt.loc[(compound, water_type), 'Mean Rs [mL/(day cm)]'] = round(sampling_rate_data_mean, 1)
            parameters_fresh_salt.loc[(compound, water_type), 'Std Rs [mL/(day cm)]'] = round(sampling_rate_data_std, 1)
            parameters_fresh_salt.loc[(compound, water_type), 'average R2score'] = round(mean_sampling_rate_r2, 2)
            parameters_fresh_salt.loc[(compound, water_type), 'average MAE [mL/(day cm)]'] = round(mean_sampling_rate_mae, 1)
            parameters_fresh_salt.loc[(compound, water_type), 'average RMSE [mL/(day cm)]'] = round(mean_sampling_rate_rmse, 1)
            parameters_fresh_salt.loc[(compound, water_type), 'Number of points'] = len(compound_water_type_data)
            parameters_fresh_salt.loc[(compound, water_type), 'Rs15 [mL/(day cm)]'] = round(sampling_rate, 1)
            parameters_fresh_salt.loc[(compound, water_type), 'Delta Rs15 [mL/(day cm)]'] = round(delta_sampling_rate, 1)

            model_sampling_rate_r2 = r2_score(
                compound_water_type_data['Sampling Rate Measured [mL/(day cm)]'],
                compound_water_type_data['Sampling Rate Model [mL/day]'] / pfas_data_class.effective_length
            )
            model_sampling_rate_mae = mean_absolute_error(
                compound_water_type_data['Sampling Rate Measured [mL/(day cm)]'],
                compound_water_type_data['Sampling Rate Model [mL/day]'] / pfas_data_class.effective_length
            )
            model_sampling_rate_rmse = root_mean_squared_error(
                compound_water_type_data['Sampling Rate Measured [mL/(day cm)]'],
                compound_water_type_data['Sampling Rate Model [mL/day]'] / pfas_data_class.effective_length
            )

            parameters_fresh_salt.loc[(compound, water_type), 'model R2score'] = round(model_sampling_rate_r2, 2)
            parameters_fresh_salt.loc[(compound, water_type), 'model MAE [mL/(day cm)]'] = round(model_sampling_rate_mae, 1)
            parameters_fresh_salt.loc[(compound, water_type), 'model RMSE [mL/(day cm)]'] = round(model_sampling_rate_rmse, 1)

            if len(time) >= 14:
                _, p_value = f_regression(
                    (1 / compound_water_type_data['Concentration Time']).values.reshape(-1, 1),
                    (compound_water_type_data['Sampling Rate Measured [mL/(day cm)]'])
                )
                parameters_fresh_salt.loc[(compound, water_type), 'inverse linear p-value'] = round(p_value[0], 3)
                if p_value[0] < 0.05:
                    # inverse linear fit
                    R_inverse_fit = LinearRegression().fit(
                        (1 / compound_water_type_data['Concentration Time']).values.reshape(-1, 1),
                        (compound_water_type_data['Sampling Rate Measured [mL/(day cm)]']).to_list()
                    )
                    gap = get_conf_int(
                        0.32, R_inverse_fit, X=pd.DataFrame((1 / compound_water_type_data['Concentration Time']).values.reshape(-1, 1)),
                        y=pd.DataFrame((compound_water_type_data['Sampling Rate Measured [mL/(day cm)]']).to_list())
                    )
                    
                    R_predicted = R_inverse_fit.predict((1 / compound_water_type_data['Concentration Time']).values.reshape(-1, 1))
                    sampling_rate_r2 = r2_score(
                        compound_water_type_data['Sampling Rate Measured [mL/(day cm)]'], R_predicted
                        )
                    sampling_rate_mae = mean_absolute_error(
                        compound_water_type_data['Sampling Rate Measured [mL/(day cm)]'], R_predicted
                        )
                    sampling_rate_rmse = root_mean_squared_error(
                        compound_water_type_data['Sampling Rate Measured [mL/(day cm)]'], R_predicted
                        )

                    parameters_fresh_salt.loc[(compound, water_type), 'inverse linear slope [mL/(ng day cm)]'] = round(R_inverse_fit.coef_[0], 1)
                    parameters_fresh_salt.loc[(compound, water_type), 'inverse Std slope [mL/(ng day cm)]'] = round(gap[1], 1)
                    parameters_fresh_salt.loc[(compound, water_type), 'inverse linear intercept [mL/(day cm)]'] = round(R_inverse_fit.intercept_, 1)
                    parameters_fresh_salt.loc[(compound, water_type), 'Delta inverse Rs [mL/(day cm)]'] = round(gap[0] + (abs(gap[1]) / compound_water_type_data['Concentration Time'].mean()), 1)
                    parameters_fresh_salt.loc[(compound, water_type), 'inverse Std intercept [mL/(day cm)]'] = round(gap[0], 1)
                    parameters_fresh_salt.loc[(compound, water_type), 'inverse linear R2score'] = round(sampling_rate_r2, 2)
                    parameters_fresh_salt.loc[(compound, water_type), 'inverse linear MAE [mL/(day cm)]'] = round(sampling_rate_mae, 1)
                    parameters_fresh_salt.loc[(compound, water_type), 'inverse linear RMSE [mL/(day cm)]'] = round(sampling_rate_rmse, 1)

                # get Ksw and standard deviation for compound in hlb
                [high_Ksw, Ksw, low_Ksw] = pfas_data_class.get_log_partition_coefficient_pfas_in_hlb(compound=compound)
                if not pd.isna(low_Ksw):
                    popt, pcov = curve_fit(concentration_exponential, time, cs_cw_ratio, p0=[sampling_rate, Ksw])
                else:
                    popt, pcov = curve_fit(concentration_exponential, time, cs_cw_ratio, p0=[15, 10**4])
                gap = np.sqrt(np.diag(pcov))
                # exponential fit
                parameters_fresh_salt.loc[(compound, water_type), 'Rs exponential coefficient [mL/(day cm)]'] = round(popt[0], 1)
                parameters_fresh_salt.loc[(compound, water_type), 'Std exponential coefficient [mL/(day cm)]'] = round(gap[0], 1)
                parameters_fresh_salt.loc[(compound, water_type), 'log(KD) pre-exponential factor [mL/(g cm)]'] = round(np.log10(popt[1]), 1)
                parameters_fresh_salt.loc[(compound, water_type), 'log(Std exponential coefficient) [mL/(g cm)]'] = round(np.log10(gap[1]), 1)
                exp_prediction = [concentration_exponential(x, popt[0], popt[1]) for x in time]
                exp_sampling_rate_r2 = r2_score(
                    compound_water_type_data['Sampling Rate Measured [mL/(day cm)]'],
                    exp_prediction / compound_water_type_data['Deployment Time [Days]']
                    )
                exp_sampling_rate_mae = mean_absolute_error(
                    compound_water_type_data['Sampling Rate Measured [mL/(day cm)]'],
                    exp_prediction / compound_water_type_data['Deployment Time [Days]']
                    )
                exp_sampling_rate_rmse = root_mean_squared_error(
                    compound_water_type_data['Sampling Rate Measured [mL/(day cm)]'],
                    exp_prediction / compound_water_type_data['Deployment Time [Days]']
                    )
                parameters_fresh_salt.loc[(compound, water_type), 'exponential R2score'] = round(exp_sampling_rate_r2, 2)
                parameters_fresh_salt.loc[(compound, water_type), 'exponential MAE [mL/(day cm)]'] = round(exp_sampling_rate_mae, 1)
                parameters_fresh_salt.loc[(compound, water_type), 'exponential RMSE [mL/(day cm)]'] = round(exp_sampling_rate_rmse, 1)

                # plot exponetial fit for PFBA
                if compound == 'PFBA':
                    ax.scatter(time, cs_cw_ratio, c=water_type_colors[water_type], marker='*', label="data")

                    if not pd.isna(low_Ksw):
                        value_vector = np.linspace(2, 56, 20)
                        ax.plot(
                            value_vector, [concentration_exponential(x, popt[0], popt[1]) for x in value_vector],
                            color=water_type_colors[water_type], linestyle='dashed', label=f'fit: ({
                                popt[1] * pfas_data_class.sorbent_weight:.2e
                                } $\cdot$ (1 - exp((-{round(popt[0], 1)} $\cdot$ $\\tau$) / ({
                                    popt[1] * pfas_data_class.sorbent_weight:.2e
                                })) mL/cm'
                                )   
                        if water_type == 'salt':
                        # plot exponential fit        
                            ax.plot(
                                value_vector, high_Ksw * pfas_data_class.sorbent_weight / pfas_data_class.effective_length * (1- np.exp(
                                    (-1) * (sampling_rate - delta_sampling_rate) * value_vector \
                                        / (pfas_data_class.sorbent_weight * high_Ksw / pfas_data_class.effective_length))
                                    ), color='k', linestyle='dotted', label='exponential model - $\\Delta$, high Ksw'
                                    )
                            ax.plot(
                                value_vector, low_Ksw * pfas_data_class.sorbent_weight / pfas_data_class.effective_length * (1- np.exp(
                                    (-1) * (sampling_rate + delta_sampling_rate) * value_vector \
                                        / (pfas_data_class.sorbent_weight * low_Ksw / pfas_data_class.effective_length))
                                    ), color='k', linestyle='dotted', label='exponential model + $\\Delta$, low Ksw'
                                    )

                            ax.set_ylabel('Cumulative Sampling Volume [mL/cm]')
                            ax.set_xlabel('Deployment Time [days]')
                            ax.legend()
                            plt.tight_layout()
                            fig.savefig(f'plots/{compound_name_plot}_sampling_rate_over_time')

In [ ]:
# average sampling rates per compound, water type, and temperature
# plots created not saved, data saved to sampling_rate_averages.csv
indices = []
for pfas_compound in pe_tube['Compound'].unique():
    for water_type in ['fresh', 'salt']:
        for temperature_level in pe_tube['Temperature Level'].unique():
            indices.append((pfas_compound, water_type, temperature_level))

sampling_rate_averages = pd.DataFrame(
    columns = [
        'Mean Rs [mL/(day cm)]', 'Std Rs [mL/(day cm)]',
        'Rs Model [mL/(day cm)]', 'Delta Rs Model [mL/(day cm)]', 'Rs15 [mL/(day cm)]',
        'average R2score', 'average MAE [mL/cm]', 'average RMSE [mL/cm]',
        ],
    index = pd.MultiIndex.from_tuples(indices)
    )

for water_type in ['fresh', 'salt']:
    # extract data for selected compound
    water_type_data = pe_tube.loc[pe_tube['Water Type'] == water_type, :]
    water_type_data['Compound'] = water_type_data['Compound'].cat.remove_unused_categories()

    number_of_points_water_type = water_type_data.groupby(['Compound'])['Sampling Rate Measured [mL/(day cm)]'].size()

    # get modelled sampling rates
    compound_indices = number_of_points_water_type.index.tolist()
    number_of_points_water_type = number_of_points_water_type.to_list()
    compound_indices = [elem for (elem, flag) in zip(compound_indices, number_of_points_water_type) if flag !=0]

    sampling_rate_model_5, delta_sampling_rate_model_5 = pfas_data_class.evaluate_sampling_rates_compound_list(
        compound_list=compound_indices, temperature_in_C=5, deltaT=10, area_correction=1/5
    )
    sampling_rate_model_15, delta_sampling_rate_model_15 = pfas_data_class.evaluate_sampling_rates_compound_list(
        compound_list=compound_indices, temperature_in_C=15, deltaT=10, area_correction=1/5
    )
    sampling_rate_model_25, delta_sampling_rate_model_25 = pfas_data_class.evaluate_sampling_rates_compound_list(
        compound_list=compound_indices, temperature_in_C=25, deltaT=10, area_correction=1/5
    )

    number_of_points_water_type = water_type_data.groupby(['Compound'])['Sampling Rate Measured [mL/(day cm)]'].size()
    mean_5C = water_type_data.loc[water_type_data['Temperature Level'] == 'cold'].groupby(
        ['Compound']
        )['Sampling Rate Measured [mL/(day cm)]'].mean()
    mean_15C = water_type_data.loc[water_type_data['Temperature Level'] == 'moderate'].groupby(
        ['Compound']
        )['Sampling Rate Measured [mL/(day cm)]'].mean()
    mean_25C = water_type_data.loc[water_type_data['Temperature Level'] == 'warm'].groupby(
        ['Compound']
        )['Sampling Rate Measured [mL/(day cm)]'].mean()

    # get modelled sampling rates
    compound_indices = number_of_points_water_type.index.tolist()
    number_of_points_water_type = number_of_points_water_type.to_list()
    compound_indices = [elem for (elem, flag) in zip(compound_indices, number_of_points_water_type) if flag !=0]
    mean_5C = [elem for (elem, flag) in zip(mean_5C, number_of_points_water_type) if flag !=0]
    mean_15C = [elem for (elem, flag) in zip(mean_15C, number_of_points_water_type) if flag !=0]
    mean_25C = [elem for (elem, flag) in zip(mean_25C, number_of_points_water_type) if flag !=0]

    ### plots to show dependencies of sampling rates: concentration, temperature, site, deployment time
    fig, ax = plt.subplots()
    water_type_data.boxplot(column='Sampling Rate Measured [mL/(day cm)]', by='Compound', ax=ax)
    ax.scatter(ax.get_xticks(), mean_5C, color='blue')
    ax.scatter(ax.get_xticks(), mean_15C, color='purple')
    ax.scatter(ax.get_xticks(), mean_25C, color='red')

    ax.plot(ax.get_xticks(), np.array(sampling_rate_model_5), color='blue', linestyle='dotted', label='R_S at 5°C')
    ax.plot(ax.get_xticks(), np.array(sampling_rate_model_15), color='purple', linestyle='dotted', label='R_S at 15°C')
    ax.plot(ax.get_xticks(), np.array(sampling_rate_model_25), color='red', linestyle='dotted', label='R_S at 25°C')

    ax.set_ylabel('Sampling Rate [mL/(day cm)]')
    box = ax.get_position()
    ax.tick_params(axis='x', rotation=90)
    ax.set_xlabel('')
    plt.show()

    for (index, compound) in enumerate(water_type_data['Compound'].unique()):
        compound_name_plot = compound.replace(':', '_')
        # extract data for selected compound
        water_type_data_compound = water_type_data.loc[water_type_data['Compound'] == compound, :]
        if len(water_type_data_compound) == 0:
            continue
        for temperature_level in water_type_data['Temperature Level'].unique():
            temperature_data = water_type_data_compound.loc[water_type_data_compound['Temperature Level'] == temperature_level, :]
            if len(temperature_data) == 0:
                continue
            if temperature_level == 'cold':
                temperature = 5
            elif temperature_level == 'moderate':
                temperature = 15
            elif temperature_level == 'warm':
                temperature = 25

            sampling_rate_data_mean = temperature_data['Sampling Rate Measured [mL/(day cm)]'].mean()
            sampling_rate_data_std = temperature_data['Sampling Rate Measured [mL/(day cm)]'].std()
            mean_sampling_rate_r2 = r2_score(cs_cw_ratio, sampling_rate_data_mean * time)
            mean_sampling_rate_mae = mean_absolute_error(cs_cw_ratio, sampling_rate_data_mean * time)
            mean_sampling_rate_rmse = root_mean_squared_error(cs_cw_ratio, sampling_rate_data_mean * time)

            value_pair = np.array([time.min(), time.max()])

            # track results in dataframe
            sampling_rate_averages.loc[(compound, water_type, temperature_level), 'Mean Rs [mL/(day cm)]'] = round(sampling_rate_data_mean, 1)
            sampling_rate_averages.loc[(compound, water_type, temperature_level), 'Std Rs [mL/(day cm)]'] = round(sampling_rate_data_std, 1)
            sampling_rate_averages.loc[(compound, water_type, temperature_level), 'Rs Model [mL/(day cm)]'] = round(sampling_rate_model, 1)
            sampling_rate_averages.loc[(compound, water_type, temperature_level), 'Delta Rs Model [mL/(day cm)]'] = round(delta_sampling_rate_model, 1)
            sampling_rate_averages.loc[(compound, water_type, temperature_level), 'average R2score'] = round(mean_sampling_rate_r2, 2)
            sampling_rate_averages.loc[(compound, water_type, temperature_level), 'average MAE [mL/(day cm)]'] = round(mean_sampling_rate_mae, 1)
            sampling_rate_averages.loc[(compound, water_type, temperature_level), 'average RMSE [mL/cm]'] = round(mean_sampling_rate_rmse, 1)
            sampling_rate_averages.loc[(compound, water_type, temperature_level), 'Number of points'] = len(temperature_data)

# write mean sampling rates to csv
sampling_rate_averages.to_csv('sampling_rate_averages.csv', index=True)

In [ ]:
# plot sampling rates for fresh and salt water alongside evaluated parameters
# save reference parameters to fitting parameters dataframe and write fitting parameters data frame to csv
for water_type in ['fresh', 'salt']:
    # extract data for selected compound
    water_type_data = pe_tube.loc[pe_tube['Water Type'] == water_type, :]
    water_type_data['Compound'] = water_type_data['Compound'].cat.remove_unused_categories()
    # get number of data points per compound for selected water type
    number_of_points_water_type = water_type_data.groupby(['Compound'])['Sampling Rate Measured [mL/(day cm)]'].size()
    # set boxplot color to water type color
    for prop in ['boxprops', 'medianprops', 'whiskerprops', 'capprops']:
        boxplot_props[prop]['color'] = water_type_colors[water_type]

    # get modelled sampling rates
    compound_indices = number_of_points_water_type.index.tolist()
    number_of_points_water_type = number_of_points_water_type.to_list()
    compound_indices = [elem for (elem, flag) in zip(compound_indices, number_of_points_water_type) if flag !=0]

    sampling_rate_model_15, delta_sampling_rate_model_15 = pfas_data_class.evaluate_sampling_rates_compound_list(
        compound_list=compound_indices, temperature_in_C=15, deltaT=10, area_correction=1/5
    )

    ### plots to show dependencies of sampling rates: concentration, temperature, site, deployment time
    fig, ax = plt.subplots()
    water_type_data.boxplot(column='Sampling Rate Measured [mL/(day cm)]', by='Compound', ax=ax, **boxplot_props)
    ax.plot(
        ax.get_xticks(), np.array(sampling_rate_model_15) + np.array(delta_sampling_rate_model_15),
        color='k', linestyle='dotted', label='R_S + $\\Delta$ R_S'
        )
    ax.plot(
        ax.get_xticks(), np.array(sampling_rate_model_15) - np.array(delta_sampling_rate_model_15),
        color='k', linestyle='dotted', label='R_S - $\\Delta$ R_S'
        )
    
    ax.set_ylabel('Sampling Rate [mL/(day cm)]')
    box = ax.get_position()
    ax.tick_params(axis='x', rotation=90)
    ax.set_xlabel('')

    legend_elements = [
        Line2D([0], [0], marker='o', markersize=6, markerfacecolor='none',
            markeredgecolor=water_type_colors[water_type], markeredgewidth=1.5,
            linestyle='none', label='Mean'),
        Line2D([0], [0], marker='*', markerfacecolor='none',
            markeredgecolor=water_type_colors[water_type], markersize=13, markeredgewidth=1.5,
            linestyle='none', label='Exponential Fit (PFBA)'),
        Line2D([0], [0], marker='^', markerfacecolor='none',
            markeredgecolor=water_type_colors[water_type], markeredgewidth=1.5,
            linestyle='none', label='Offset corrected'),
        Patch(edgecolor=water_type_colors[water_type], fill=False, label='Boxplot'),
        Line2D([0], [0], marker='.', color='w', markeredgecolor='gray', markersize=6, linestyle='none',
            label='Outlier',),
        Line2D([0], [0], color='k', linestyle='dotted', label='Model bounds',)
            ]

    if water_type == 'fresh':
        legend_elements.append(
            Line2D([0], [0], marker='.', color='w', markersize=10, 
                markerfacecolor='blue', label='Reference Spain')
            )
        legend_elements.append(
            Line2D([0], [0], marker='*', color='w', markersize=10, 
                markerfacecolor='blue', label='Reference Australia')
            )

    ax.legend(
        handles=legend_elements, loc="lower center",
        bbox_to_anchor=(0.5, 1.02), fancybox=True, ncol=3
        )

    for (index, compound) in enumerate(water_type_data['Compound'].unique()):
        compound_name_plot = compound.replace(':', '_')
        # extract data for selected compound
        water_type_data_compound = water_type_data.loc[water_type_data['Compound'] == compound, :]
        if len(water_type_data_compound) == 0:
            continue
        if compound == 'PFBA':
            Rs = parameters_fresh_salt.loc[(compound, water_type), 'Rs exponential coefficient [mL/(day cm)]']
            deltaRs = parameters_fresh_salt.loc[(compound, water_type), 'Std exponential coefficient [mL/(day cm)]']
            ax.scatter(x=index + 1, y=Rs, marker='*', facecolors='none',
                edgecolors=water_type_colors[water_type], s=80, linewidth=1.5,)
        if water_type == 'fresh':
            for (ref_loc, marker) in zip(sampling_rates_ref_fresh.keys(), ['.', '*']):
                if compound in list(sampling_rates_ref_fresh[ref_loc]['reference Mean Rs [mL/(day cm)]'].keys()):
                    ax.scatter(
                        x=index + 1,
                        y=sampling_rates_ref_fresh[ref_loc]['reference Mean Rs [mL/(day cm)]'][compound],
                        marker=marker,
                        color='blue'
                        )
                    if ref_loc == 'spain':
                        parameters_fresh_salt.loc[(compound, water_type), 'ref Spain Mean Rs [mL/(day cm)]'] = \
                            sampling_rates_ref_fresh[ref_loc]['reference Mean Rs [mL/(day cm)]'][compound]
                    if ref_loc == 'australia':
                        parameters_fresh_salt.loc[(compound, water_type), 'ref Australia Mean Rs [mL/(day cm)]'] = \
                            sampling_rates_ref_fresh[ref_loc]['reference Mean Rs [mL/(day cm)]'][compound]
        # plot mean sampling rate with error bars
        Rs = parameters_fresh_salt.loc[(compound, water_type), 'Mean Rs [mL/(day cm)]']
        deltaRs = parameters_fresh_salt.loc[(compound, water_type), 'Std Rs [mL/(day cm)]']
        ax.scatter(x=index + 1, y=Rs, marker = 'o', facecolors='none',
                   edgecolors=water_type_colors[water_type], s=40, linewidth=1.5,)
        # plot inverse linear fit intercept as Rs with error bars
        Rs = parameters_fresh_salt.loc[(compound, water_type), 'inverse linear intercept [mL/(day cm)]']
        deltaRs = parameters_fresh_salt.loc[(compound, water_type), 'Delta inverse Rs [mL/(day cm)]']
        slope = parameters_fresh_salt.loc[(compound, water_type), 'inverse linear slope [mL/(ng day cm)]']
        if (not np.isnan(Rs)) and slope >= 0 and Rs >= 0:  #and water_type == 'fresh'
            ax.scatter(x=index + 1, y=Rs, marker = '^', facecolors='none',
                       edgecolors=water_type_colors[water_type], s=40, linewidth=1.5,)

    add_counts_to_xticklabels(ax, number_of_points_water_type)

    fig.suptitle('')
    plt.tight_layout()
    ax.set_title('')
    ax.yaxis.grid(True)
    fig.savefig(f'plots/{water_type}_per_compound.png')

selected_columns = [
    'Mean Rs [mL/(day cm)]', 'inverse linear intercept [mL/(day cm)]',
    'Rs15 [mL/(day cm)]', 'ref Spain Mean Rs [mL/(day cm)]', 'ref Australia Mean Rs [mL/(day cm)]',
]
selected_columns_PFBA = [
    'Mean Rs [mL/(day cm)]', 'Rs exponential coefficient [mL/(day cm)]',
    'Rs15 [mL/(day cm)]', 'ref Spain Mean Rs [mL/(day cm)]', 'ref Australia Mean Rs [mL/(day cm)]',
]

parameters_fresh_salt['coefficient of variation [%]'] = ((
    parameters_fresh_salt[selected_columns].std(axis=1) / parameters_fresh_salt[selected_columns].mean(axis=1)
    ) * 100).astype(float).round(decimals=0)
parameters_fresh_salt.at[('PFBA', 'fresh'), 'coefficient of variation [%]'] = round((
    parameters_fresh_salt.loc[('PFBA', 'fresh'), selected_columns_PFBA].std() / parameters_fresh_salt.loc[('PFBA', 'fresh'), selected_columns_PFBA].mean()
    ) * 100, 0)

# write fitting parameters to csv
parameters_fresh_salt.to_csv('parameters_fresh_salt.csv', index=True)

In [ ]:
# evaluate statistical significance:
manova_data = pe_tube.rename(columns={
    'Sampling Rate Measured [mL/day]': 'SR',
    'Water Type': 'WT',
    'Functional Group': 'FG',
    'Deployment Time': 'DT', 
    'Carbon Chain Length': 'CCL',
    'Average Temperature in C': 'T',
    'Grab Sample Average': 'GA'})

# convert days to float
manova_data['DT'] = manova_data['DT'].dt.days
manova_data['sqrt_SR'] = np.sqrt(manova_data['SR'])

# get summary of number of data points per deployment time and water type
dt_count = []
for water_type in ['ground', 'fresh', 'salt']:
    dt_count.append(manova_data[manova_data['WT'] == water_type].groupby('DT').size())
dt_summary = pd.concat(dt_count, axis=1).fillna(0).astype(int).rename(
    columns={0: 'ground', 1: 'fresh', 2: 'salt'}
    )
dt_summary.sort_index(inplace=True)
dt_summary.to_csv('deployment_time_summary.csv')

# ANOVA for fresh and salt water separately
fig_qq, ax_qq = plt.subplots()
fig_hist, ax_hist = plt.subplots()
for water_type in ['fresh', 'salt']:
    manova_data_water_type = manova_data[manova_data['WT'] == water_type]
    scaler = StandardScaler()
    # perform three-way ANOVA
    if water_type == 'ground':
        manova_data_water_type[['sqrt_SR', 'CCL']] = scaler.fit_transform(manova_data_water_type[['sqrt_SR', 'CCL']])
        model = ols("""sqrt_SR ~ CCL : C(FG)""", data=manova_data_water_type).fit()
    else:
        manova_data_water_type[['DT', 'GA', 'T', 'sqrt_SR', 'CCL']] = scaler.fit_transform(manova_data_water_type[['DT', 'GA', 'T', 'sqrt_SR', 'CCL']])
        model = ols("""sqrt_SR ~ CCL : C(FG) + DT + GA + T""", data=manova_data_water_type).fit()
    print(f"Results for {water_type} water:")
    print(sm.stats.anova_lm(model, typ=2))

    sm.qqplot(
        ax=ax_qq, data=model.resid, line='45', fit=True, label=water_type,
        markerfacecolor=water_type_colors[water_type], markeredgecolor=water_type_colors[water_type],
        )
    plt.hist(model.resid, bins=100, label=water_type, color=water_type_colors[water_type])
    ax_hist.legend()
    ax_hist.set_xlabel('Residuals')
    ax_hist.set_ylabel('Counts')

    for functional_group in ['CO2H', 'SO3H', 'SO2NH2']:
        manova_data_water_type_functional_group = manova_data[manova_data['FG'] == functional_group]
        manova_data_water_type_functional_group[['sqrt_SR', 'CCL']] = scaler.fit_transform(manova_data_water_type_functional_group[['sqrt_SR', 'CCL']])
        model = ols("""sqrt_SR ~ CCL + DT + GA + T""", data=manova_data_water_type_functional_group).fit()
        print(f"Results for {water_type} water and {functional_group} functional group:")
        print(sm.stats.anova_lm(model, typ=2))
        sm.qqplot(
        ax=ax_qq, data=model.resid, line='45', fit=True, label=water_type,
        markerfacecolor=water_type_colors[water_type], markeredgecolor=water_type_colors[water_type],
        )
        plt.show()
        plt.hist(model.resid, bins=100, label=water_type, color=water_type_colors[water_type])
        plt.show()

    # # MANOVA - does not reveal more than ANOVA, so commented out.
    # print('basics only')
    # manova = MANOVA.from_formula('CCL : C(FG) ~ sqrt_SR', data=manova_data_water_type)
    # result = manova.mv_test()
    # print(result.summary())

    # print('basics only')
    # manova = MANOVA.from_formula('CCL : C(FG) + DT ~ sqrt_SR', data=manova_data_water_type)
    # result = manova.mv_test()
    # print(result.summary())

    # print('basics + temperature')
    # manova = MANOVA.from_formula('CCL : C(FG) + DT + T ~ sqrt_SR', data=manova_data_water_type)
    # result = manova.mv_test()
    # print(result.summary())

    # print('basics + c_W')
    # manova = MANOVA.from_formula('CCL : C(FG) + DT + GA ~ sqrt_SR', data=manova_data_water_type)
    # result = manova.mv_test()
    # print(result.summary())


fig_hist.savefig(f'plots/ANOVA_residuals_hist.png')
fig_qq.savefig(f'plots/ANOVA_residuals_qq.png')


In [ ]:
# plot sampling rates over time and over concentration time for selected compounds
fig_exp, ax_exp = plt.subplots(ncols=2, nrows=5, sharex=True, sharey=True, figsize=(12, 16))

for water_type in ['fresh', 'salt']:
    fig, ax = plt.subplots(ncols=2, nrows=5, sharey=True, figsize=(12, 16))
    water_type_data = pe_tube.loc[pe_tube['Water Type']==water_type,:]
    water_type_data['Deployment Time [days]'] = water_type_data['Deployment Time'].dt.days

    for prop in ['boxprops', 'medianprops', 'whiskerprops', 'capprops']:
        boxplot_props[prop]['color'] = water_type_colors[water_type]
    for (compound_index, compound) in enumerate([
        'PFBA', 'PFBS', 'PFPeA', 'PFPeS', 'PFHxA', 'PFHxS', 'PFHpA',  'PFHpS', 'PFOA', 'PFOS'
    ]):
        ncol = compound_index % 2
        nrow = compound_index // 2

        water_type_compound_data = water_type_data[water_type_data['Compound']==compound]
        x_label = np.sort(water_type_compound_data['Deployment Time [days]'].unique())
        water_type_compound_data.boxplot(
            column=['Sampling Rate Measured [mL/(day cm)]'], by='Deployment Time [days]', ax=ax[nrow, ncol],
            positions=x_label, **boxplot_props,
            )
        number_of_points_site = water_type_compound_data.groupby('Deployment Time [days]')[
                'Sampling Rate Measured [mL/(day cm)]'
                ].size().to_list()
        median = water_type_compound_data.groupby('Deployment Time [days]')[
            'Sampling Rate Measured [mL/(day cm)]'
            ].median().to_list()

        for (x_tick_index, x_tick_value) in enumerate(ax[nrow, ncol].get_xticks()):
            ax[nrow, ncol].text(
                x_tick_value, median[x_tick_index] + 1, number_of_points_site[x_tick_index],
                horizontalalignment='center', color='blue', weight='bold',
                )
        sampling_rate_15, delta_sampling_rate_15 = pfas_data_class.evaluate_sampling_rate(
            compound=compound, temperature_in_C=15, deltaT=10, area_correction=1/pfas_data_class.effective_length
            )
        ax[nrow, ncol].plot(
            [x_label[0] - 1, x_label[-1] + 1], [sampling_rate_15 + delta_sampling_rate_15] * 2,
            color='k', linestyle='dotted', linewidth=3, label='model R_S + $\\Delta$ R_S'
            )
        ax[nrow, ncol].plot(
            [x_label[0] - 1, x_label[-1] + 1], [sampling_rate_15 - delta_sampling_rate_15] * 2,
            color='k', linestyle='dotted', linewidth=3, label='model R_S - $\\Delta$ R_S'
            )
            
        if water_type == 'fresh':
            ax[nrow, ncol].set_xlim([1, 52])
        elif water_type == 'salt':
            ax[nrow, ncol].set_xlim([13, 57])
        ax[nrow, ncol].set_title(compound)
        # add_counts_to_xticklabels(ax[nrow, ncol], number_of_points_site)

        if water_type_compound_data.empty:
            continue
        if parameters_fresh_salt.loc[(compound, water_type), 'inverse linear p-value'] < 0.05 and \
              not np.isnan(parameters_fresh_salt.loc[(compound, water_type), 'inverse linear p-value']):
            R_inverse_fit = LinearRegression().fit(
                (1 / water_type_compound_data['Concentration Time']).values.reshape(-1, 1),
                water_type_compound_data['Sampling Rate Measured [mL/(day cm)]'].to_list()
                )
            fake_x = np.logspace(
                np.log10(water_type_compound_data['Concentration Time'].min()),
                np.log10(water_type_compound_data['Concentration Time'].max()), 100
                )
            ax_exp[nrow, ncol].plot(
                fake_x, R_inverse_fit.predict((1 / fake_x).reshape(-1, 1)),
                color=water_type_colors[water_type], linestyle='dashed', linewidth=3,
                label=f'R_S = {round(R_inverse_fit.intercept_, 0)} + {round(R_inverse_fit.coef_[0], 0)} / ($c_W$ $\cdot$ $\\tau$)'
                )
        
        ax_exp[nrow, ncol].scatter(
            water_type_compound_data['Concentration Time'].to_list(), water_type_compound_data['Sampling Rate Measured [mL/(day cm)]'].to_list(),
            color=water_type_colors[water_type], label=water_type
        )
        if water_type == 'salt':
            ax_exp[nrow, ncol].legend()
        if water_type == 'fresh':
            ax_exp[nrow, ncol].set_xlim([0, 800])  #52
        
            value_pair = [water_type_compound_data['Concentration Time'].min(), water_type_compound_data['Concentration Time'].max()]
            ax_exp[nrow, ncol].plot(
                value_pair, [sampling_rate_15 + delta_sampling_rate_15] * 2,
                color='k', linestyle='dotted', linewidth=3, label='model R_S + $\\Delta$ R_S'
                )
            ax_exp[nrow, ncol].plot(
                value_pair, [sampling_rate_15 - delta_sampling_rate_15] * 2,
                color='k', linestyle='dotted', linewidth=3, label='model R_S - $\\Delta$ R_S'
                )
        if ncol == 0:
            ax[nrow, ncol].set_ylabel('Sampling Rate [mL/(day cm)]')
            ax_exp[nrow, ncol].set_ylabel('Sampling Rate [mL/(day cm)]')
        if nrow == 4:
            ax[nrow, ncol].tick_params(axis='x', rotation=90)
            ax_exp[nrow, ncol].tick_params(axis='x', labelbottom=True)
            ax_exp[nrow, ncol].set_xlabel('Deployment Time [days] * Grab Conc. [ng/L]')
        else:
            ax[nrow, ncol].set_xticklabels('')
            ax[nrow, ncol].set_xlabel('')

        ax_exp[nrow, ncol].set_title(compound)
            
        if ncol == 0 and nrow == 0:
            ax[nrow, ncol].legend()
        ax[nrow, ncol].yaxis.grid(True)

    fig.suptitle('')
    plt.tight_layout()
    fig.savefig(f'plots/summary_{water_type}_deployment_time', bbox_inches='tight', pad_inches=0)

fig_exp.suptitle('')
fig_exp.savefig(f'plots/sampling_rate_fitted', bbox_inches='tight', pad_inches=0)

In [ ]:
# plot sampling rates over temperature and over background concentrations for selected compounds
fig, ax = plt.subplots(ncols=2, nrows=5, sharey=True, figsize=(12, 16))
fig_conc, ax_conc = plt.subplots(ncols=2, nrows=5, sharex=True, sharey=True, figsize=(12, 16))

for (compound_index, compound) in enumerate([
    'PFBA', 'PFBS', 'PFPeA', 'PFPeS', 'PFHxA', 'PFHxS', 'PFHpA',  'PFHpS', 'PFOA', 'PFOS'
]):  
    ncol = compound_index % 2
    nrow = compound_index // 2

    temperatures = np.linspace(0, 25, 10)
    sampling_rates = []
    delta_sampling_rates = []
    for temperature in temperatures:
        sampling_rate, delta_sampling_rate = pfas_data_class.evaluate_sampling_rate(
            compound=compound, temperature_in_C=temperature, deltaT=5, area_correction=1/pfas_data_class.effective_length
            )
        sampling_rates.append(sampling_rate)
        delta_sampling_rates.append(delta_sampling_rate)

    ax[nrow, ncol].plot(
        temperatures, [t + deltat for (t, deltat) in zip(sampling_rates, delta_sampling_rates)],
        color='k', linestyle='dotted', linewidth=3, label='model R_S + $\\Delta$ R_S'
        )
    ax[nrow, ncol].plot(
        temperatures, [t - deltat for (t, deltat) in zip(sampling_rates, delta_sampling_rates)],
        color='k', linestyle='dotted', linewidth=3, label='model R_S - $\\Delta$ R_S'
        )
    
    ax_conc[nrow, ncol].plot(
        [0, 60], [sampling_rates[-1] + delta_sampling_rates[-1]] * 2,
        color='k', linestyle='dotted', linewidth=3, label='model R_S + $\\Delta$ R_S'
        )
    ax_conc[nrow, ncol].plot(
        [0, 60], [sampling_rates[0] - delta_sampling_rates[0]] * 2,
        color='k', linestyle='dotted', linewidth=3, label='model R_S - $\\Delta$ R_S'
        )

    for water_type in ['fresh', 'salt']:
        water_type_data = pe_tube[pe_tube['Water Type']==water_type]
        fresh_compound_data = water_type_data[water_type_data['Compound']==compound]
        if fresh_compound_data.empty:
            continue
        ax[nrow, ncol].scatter(
            fresh_compound_data['Average Temperature in C'].to_list(), fresh_compound_data['Sampling Rate Measured [mL/(day cm)]'].to_list(),
            color=water_type_colors[water_type], label=water_type
        )
        ax_conc[nrow, ncol].scatter(
            fresh_compound_data['Grab Sample Average'].to_list(), fresh_compound_data['Sampling Rate Measured [mL/(day cm)]'].to_list(),
            color=water_type_colors[water_type], label=water_type
        )
        ax_conc[nrow, ncol].set_xlim([0, 60])
        if ncol == 0:
            ax[nrow, ncol].set_ylabel('Sampling Rate [mL/(day cm)]')
            ax_conc[nrow, ncol].set_ylabel('Sampling Rate [mL/(day cm)]')
        if nrow == 4:
            ax[nrow, ncol].set_xlabel('Temperature [°C]')
            ax_conc[nrow, ncol].set_xlabel('Average Water Concentration [ng/L]')

        ax[nrow, ncol].set_title(compound)
        ax_conc[nrow, ncol].set_title(compound)
            
        if ncol == 0 and nrow == 0:
            ax[nrow, ncol].legend()
            ax_conc[nrow, ncol].legend()

    fig.suptitle('')
    fig_conc.suptitle('')
    fig_exp.suptitle('')
    plt.tight_layout()
    fig.savefig(f'plots/summary_temperature', bbox_inches='tight', pad_inches=0)
    fig_conc.savefig(f'plots/summary_concentration', bbox_inches='tight', pad_inches=0)

In [ ]:
# PCA
label = 'Sampling Rate Measured [mL/(day cm)]' # 'Grab Sample Average'
print(pe_tube.columns)
variable = 'Water Type'

# create longdata for pca and use sqrt of label as target (analogy to ANOVA)
pca_data = pe_tube.loc[~pe_tube[label].isna(), :]
pca_data[label] = np.sqrt(pca_data[label])
pca_data['replicate'] = pca_data.groupby(['Site', 'Compound']).cumcount() + 1
pca_data = pca_data.pivot(index=['Site', variable, 'replicate'], columns='Compound', values=label).reset_index()
pca_data_clean = pca_data.drop(columns=['Site', variable, 'replicate'])

# drop na rows with more than 30% NaN
nan_fraction = pca_data_clean.isna().mean()
pca_data_clean = pca_data_clean.loc[:, nan_fraction <= 0.3]

# replace remaining NaN with half minimum value
pca_data_clean = pca_data_clean.apply(
    lambda col: col.fillna(col.min() / np.sqrt(2)) if np.issubdtype(col.dtype, np.number) else col
)

# normalize features and depl PCA
pca_data_features = StandardScaler().fit_transform(pca_data_clean)
pca_model = PCA(n_components=2)
principal_components = pca_model.fit_transform(pca_data_features)

# PCA loadings
loadings = pd.DataFrame(pca_model.components_.T, columns=['PC1', 'PC2'], index=pca_data_clean.columns)
display(loadings)

# print relevance of PCA
print('Explained variability per principal component: {}'.format(pca_model.explained_variance_ratio_))

# combine new data
principal_df = pd.DataFrame(data = principal_components, columns = ['principal component 1', 'principal component 2'])
principal_df[variable] = pca_data[variable]

# plots
sites = principal_df[variable].unique()
arrow_size = 3.5
text_pos = 4.0
fig = plt.figure()
for i, site in enumerate(sites):
    subset = principal_df[principal_df[variable] == site]
    plt.scatter(subset['principal component 1'], subset['principal component 2'], 
                label=site, marker='*',
                color=water_type_colors[site],)
for i in range(loadings.shape[0]):
    plt.arrow(0, 0, 
              loadings.PC1[i]*arrow_size, 
              loadings.PC2[i]*arrow_size, 
              color='k', alpha=0.7, head_width=0.05)
    plt.text(loadings.PC1[i]*text_pos, 
             loadings.PC2[i]*text_pos, 
             loadings.index[i], color='k', fontsize=12)    
plt.legend(bbox_to_anchor=(0.9, -0.12), fancybox=True, ncol=4)
plt.xlabel("principal component 1")
plt.ylabel("principal component 2")
plt.tight_layout()

plt.xlabel("principal component 1")
plt.ylabel("principal component 2")
plt.tight_layout()
plt.show()
fig.savefig('plots/pca.png')

In [ ]:

fig, axes = plt.subplots(2, 2, sharey=True, figsize=(8, 8))

compounds = [index[0] for index in parameters_fresh_salt.index][::2]
x = np.arange(len(compounds))
width = 0.35

for metric_idx, metric in enumerate(['MAE [mL/(day cm)]', 'RMSE [mL/(day cm)]']):
    for water_idx, water_type in enumerate(['fresh', 'salt']):
        mean_metric = f'average {metric}'
        model_metric = f'model {metric}'
        mean = [parameters_fresh_salt.loc[(c, water_type), mean_metric] for c in compounds]
        model = [parameters_fresh_salt.loc[(c, water_type), model_metric] for c in compounds]

        axes[metric_idx, water_idx].bar(x - width/2, mean, width, label='Mean', alpha=0.8)
        axes[metric_idx, water_idx].bar(x + width/2, model, width, label='Model', alpha=0.8)
        axes[metric_idx, water_idx].set_xlabel('Compound')
        axes[metric_idx, water_idx].set_ylabel(metric)
        axes[metric_idx, water_idx].set_title(f'{metric} by Water Type')
        axes[metric_idx, water_idx].set_xticks(x)
        axes[metric_idx, water_idx].set_xticklabels(compounds)
        axes[metric_idx, water_idx].legend(title=f'{water_type} water')
        axes[metric_idx, water_idx].grid(axis='y', alpha=0.3)
        add_counts_to_xticklabels(
            axes[metric_idx, water_idx], parameters_fresh_salt.loc[(slice(None), water_type), 'Number of points'], rotation=90
            )
        axes[metric_idx, water_idx].set_title('')

plt.tight_layout()
plt.savefig('plots/mae_rmse.png', bbox_inches='tight', pad_inches=0)
plt.show()